# Optional: VIMO's understanding model

**Plain words:** VIMO already understands sentences with written rules (they stay as the safety net). This notebook trains a small extra model that maps a sentence to its intent (milk, expense, new cow…) from the corpus and the labelled sample sentences, and exports it as an `nlu` language pack. Run each cell from top to bottom.

In [ ]:
!git clone --depth 1 https://github.com/skwatsonoff/vimo-ranch-management.git
%cd vimo-ranch-management
!pip install --quiet scikit-learn skl2onnx onnxruntime

## 1. Sentences and their intents
From `assets/voice/voice_corpus.json` and `test/data/voice_intent_samples.jsonl`.

In [ ]:
import json
rows = [(u['text'], u['intent']) for u in json.load(open('assets/voice/voice_corpus.json'))['utterances']]
for line in open('test/data/voice_intent_samples.jsonl', encoding='utf-8'):
    r = json.loads(line); rows.append((r['input'], r['output']['action']))
texts, labels = zip(*rows)
print(len(rows), 'sentences,', len(set(labels)), 'intents')

## 2. Train (character n-grams work for Tamil, Tanglish and English alike)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
x_tr, x_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.15, random_state=7, stratify=labels)
model = make_pipeline(TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=2), LogisticRegression(max_iter=2000))
model.fit(x_tr, y_tr)
print('accuracy on unseen sentences:', round(model.score(x_te, y_te), 3))

## 3. Export as an `nlu` pack (ONNX)
The file `nlu-ta.onnx` can be published with `tools/voice_training/build_pack_manifest.py`. The app keeps using its rules until it learns to load this pack.

In [ ]:
from skl2onnx import to_onnx
from skl2onnx.common.data_types import StringTensorType
onx = to_onnx(model, initial_types=[('text', StringTensorType([None, 1]))])
open('nlu-ta.onnx', 'wb').write(onx.SerializeToString())
print('saved nlu-ta.onnx')